# 04 · Build Silver — Account Dimension

**Layer:** Silver &nbsp;|&nbsp; **Stage:** transformation &nbsp;|&nbsp; **Target:** `banking_silver.dim_account`

First of the two silver builds, running in parallel with
`05_build_silver_txn_clean`. Turns the raw account feed into a conformed
dimension that downstream gold aggregates can trust as a single row per
account.

**Applied here**

1. rows with a null `account_id` are dropped — they cannot be keyed
2. replayed duplicates collapsed, keeping the most recently ingested version
3. `status` upper-cased and constrained to a known set
4. `opened_on` parsed to a real `DATE`, `risk_band` defaulted where absent
5. missing `branch_code` filled with the `UNKNOWN` sentinel rather than null,
   so joins do not silently drop rows
6. derived attributes added: `account_age_years`, `is_active`, `risk_category`


In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("medallion-silver-dim-account").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"
SILVER_DB = "banking_silver"

bronze = spark.table(f"{BRONZE_DB}.raw_accounts")
print("bronze rows in:", bronze.count())

In [ ]:
# 1. drop unkeyable rows
keyed = bronze.filter(F.col("account_id").isNotNull())
print(f"dropped {bronze.count() - keyed.count()} row(s) with a null account_id")

In [ ]:
# 2. collapse replayed duplicates - keep the latest ingested version
dedupe_window = Window.partitionBy("account_id").orderBy(F.col("_ingested_at").desc())

deduped = (
    keyed
    .withColumn("_rn", F.row_number().over(dedupe_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)
print(f"collapsed {keyed.count() - deduped.count()} duplicate account row(s)")

In [ ]:
# 3-5. standardise, type, and fill sentinels
VALID_STATUS = ["ACTIVE", "DORMANT", "CLOSED", "FROZEN"]

conformed = (
    deduped
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    .withColumn(
        "status",
        F.when(F.col("status").isin(VALID_STATUS), F.col("status")).otherwise(F.lit("UNKNOWN")),
    )
    .withColumn("account_type", F.upper(F.trim(F.col("account_type"))))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("branch_code", F.coalesce(F.col("branch_code"), F.lit("UNKNOWN")))
    .withColumn("customer_id", F.coalesce(F.col("customer_id"), F.lit("UNASSIGNED")))
    .withColumn("opened_on", F.to_date(F.col("opened_on"), "yyyy-MM-dd"))
    .withColumn("opening_balance", F.round(F.coalesce(F.col("opening_balance"), F.lit(0.0)), 2))
    .withColumn("risk_band", F.coalesce(F.col("risk_band"), F.lit(3)))
)

In [ ]:
# 6. derived attributes
dim_account = (
    conformed
    .withColumn(
        "account_age_years",
        F.round(F.months_between(F.to_date(F.lit(RUN_DATE)), F.col("opened_on")) / 12, 1),
    )
    .withColumn("is_active", F.col("status") == F.lit("ACTIVE"))
    .withColumn(
        "risk_category",
        F.when(F.col("risk_band") <= 1, "LOW")
         .when(F.col("risk_band") <= 3, "MEDIUM")
         .otherwise("HIGH"),
    )
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_run_date", F.lit(RUN_DATE))
    .withColumn("_processed_at", F.current_timestamp())
    .select(
        "account_id", "customer_id", "account_type", "branch_code", "currency",
        "opening_balance", "status", "is_active", "opened_on", "account_age_years",
        "risk_band", "risk_category",
        "_batch_id", "_run_date", "_processed_at",
    )
)

dim_account.printSchema()

In [ ]:
(
    dim_account.write
    .mode("overwrite")
    .format("parquet")
    .saveAsTable(f"{SILVER_DB}.dim_account")
)

out = spark.table(f"{SILVER_DB}.dim_account")
print(f"wrote {SILVER_DB}.dim_account  ({out.count()} rows)")
out.orderBy("account_id").show(truncate=False)

In [ ]:
out.groupBy("risk_category", "status").count().orderBy("risk_category", "status").show(truncate=False)